# The Augmented LLM: From a Single Call to an Orchestrated Workflow

This notebook is a guided walkthrough of three small scripts you wrote, but instead of reading them one at a time, we're going to build up a single mental model — the same way a course would.

The organizing idea comes from Anthropic's "Building Effective Agents" framing: the **augmented LLM** is the basic unit of every agentic system. It's just an LLM call, but *augmented* with extra capabilities — tools it can invoke, a schema it must fill in, retrieval it can query. Once you have augmented LLM calls, you can wire several of them together into a **workflow** — a predefined, orchestrated sequence of steps.

Your three scripts map onto exactly this progression:

| Script | What it actually demonstrates |
|---|---|
| `augmented_llm.py` | Augmenting an LLM with **tools** |
| `augmented_llm_structured_output.py` | Augmenting an LLM with a **structured output schema** |
| `augmented_llm_prompt_chaining.py` | Composing several augmented LLM calls into a **workflow** (prompt chaining), using LangGraph as the orchestrator |

We'll go in that order: single augmented calls first, then the orchestration layer that strings them together. Along the way I'll flag real bugs/typos I noticed with **⚠️ Bug watch** callouts — I'm pointing them out rather than silently fixing them, so look at each one and see if you'd have caught it.

## 1. Setup

Every script starts the same way: load environment variables, then construct a `ChatGroq` model object.

**Why `load_dotenv()`?** Your Groq API key shouldn't be hardcoded into source files (that's how keys end up leaked on GitHub). Instead, it lives in a local `.env` file (e.g. `GROQ_API_KEY=...`) which is `.gitignore`d. `load_dotenv()`, from `python-dotenv`, reads that file and injects its contents into `os.environ` *before* any library that needs the key gets imported/used. `ChatGroq` picks the key up implicitly from the environment — you never pass it explicitly in any of your scripts, which is the intended pattern.

**Why `ChatGroq` and not, say, `ChatOpenAI`?** LangChain ships a family of chat-model wrapper classes, one per provider, all implementing the same interface (`.invoke()`, `.bind_tools()`, `.with_structured_output()`, etc.). This is the point of LangChain's abstraction: swapping providers is (in principle) a one-line change because the rest of your code — prompts, chains, graphs — never touches the provider-specific API directly.

`openai/gpt-oss-120b` is the specific open-weight model you're targeting on Groq's fast inference infrastructure — the name is a Groq/OpenAI-open-weights model id, not an OpenAI-hosted model.

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")

# The bare, unaugmented call: no tools, no schema, just text in, text out.
response = llm.invoke("In one sentence, what is a large language model?")
print(response.content)

That call above is the *plain* LLM — no augmentation. `response` is an `AIMessage` object; `.content` is the text. Everything from here is about extending what that single call can do.

## 2. Augmentation #1 — Giving the LLM tools

An LLM can't check today's weather or do arithmetic reliably — it can only generate text. **Tool calling** solves this: you describe some Python functions to the model, and instead of forcing it to answer directly, the model can respond with "call this function with these arguments" and let *your code* execute it.

### The `@tool` decorator

LangChain's `@tool` decorator turns an ordinary Python function into something the model can be told about. Two details matter a lot here, because the LLM never sees your source code — it only sees a name, an argument schema, and **the docstring**:

- The **docstring becomes the tool's description**. If it's vague or wrong, the model will call the tool at the wrong times or misunderstand what it returns.
- Type hints (`city: str`, `bill_amount: float`) become the tool's argument schema (via Pydantic under the hood), so the model knows what types of arguments to produce.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Get the weather for a city
    Arg:
        city : Name of the city
    """
    weather_data = {
        "New York": "Sunny, 72 F",
        "London": "Cloudy 15 F",
        "Tokyo": "Rainy, 20 C",
    }
    return weather_data.get(city, "Weather data not available")


@tool
def calculate_tip(bill_amount: float, tip_percentage: float) -> float:
    """
    Calculate tip amount based on the bill and the percentage
    Arg:
        bill_amount : The original bill amount to be paid
        tip_percentge: The percentage of the bill amount that is to be paid extra as the tip
    """
    return round(bill_amount * (tip_percentage / 100), 2)

**⚠️ Bug watch — typo in a docstring:** `calculate_tip`'s docstring says `tip_percentge`, not `tip_percentage`. This isn't a Python error (docstrings are just strings), but it *is* a real quality bug: this text is what the LLM reads to understand the parameter, and a misspelled name in the *description* right next to the correctly-spelled real parameter can be a source of confusion for the model, and is definitely confusing for a human reading the code later.

**⚠️ Bug watch — inconsistent data, not a bug but worth a second look:** `weather_data` mixes units without labeling them consistently — New York and London are given in Fahrenheit, Tokyo in Celsius, and London's string oddly reads "Cloudy 15 F" where 15°F is bitterly cold (more plausibly this was meant to be 15°C). It won't crash anything, but if you were building something real on top of this, silently-wrong units are the kind of bug that's invisible until someone acts on the number.

### Binding tools and reading `tool_calls`

`llm.bind_tools([...])` returns a *new* runnable — it doesn't mutate `llm` — that has the tool schemas attached to every request sent to the model. When invoked, the model can choose to respond with one or more **tool calls** instead of (or alongside) plain text; these show up in `response.tool_calls`, a list of dicts with the tool name and arguments the model decided on.

In [ ]:
llm_with_tools = llm.bind_tools([get_weather, calculate_tip])

weather_prompt = "What's the weather like in Tokyo?"
tip_prompt = "Calculate a 20 tip on a 50 bill"

response = llm_with_tools.invoke(weather_prompt)
tool_calls_1 = response.tool_calls

tool_calls_2 = llm_with_tools.invoke(tip_prompt).tool_calls

print(tool_calls_1)
print("=" * 69)
print(tool_calls_2)

**Important gap to notice:** this code stops at *deciding* to call a tool. Nothing in the script actually runs `get_weather(...)` or `calculate_tip(...)` with the arguments the model chose, and nothing feeds a result back to the model for a final answer. That's fine — the script's job was just to demonstrate tool binding — but a complete tool-use loop needs one more step: execute each tool call, wrap the result in a `ToolMessage`, and send the conversation (original prompt + AI's tool call + tool result) back to the model for a final natural-language answer. That full loop is what frameworks like LangGraph's prebuilt ReAct agent do for you automatically. We'll touch this again in the exercises.

Also worth knowing: tool calling isn't guaranteed. The model *decides* whether a prompt warrants a tool call; for an ambiguous prompt it might just answer in text and `tool_calls` would be an empty list. Production code should check for that rather than assuming `tool_calls` is populated.

## 3. Augmentation #2 — Giving the LLM a schema (structured output)

Tools let the model *act*. Structured output lets the model *report back* in a shape your program can rely on, instead of prose you'd have to parse with regex and hope for the best.

### Defining the schema with Pydantic

A `pydantic.BaseModel` subclass is both documentation and a runtime-validated contract. Each `Field(description=...)` isn't just for humans — like the tool docstrings above, these descriptions are sent to the model as instructions for what to put in each slot.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ProductReview(BaseModel):
    """
    Structured Product review analysis
    """
    product_name: str = Field(description="Name of the product")
    sentiment: str = Field(description="Overall sentiment of the: positive, negative or neutral")
    rating: int = Field(description="Rating from 1-5", ge=1, le=5)
    pros: List[str] = Field(description="List of the positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

`ge=1, le=5` (greater-than-or-equal / less-than-or-equal) are Pydantic validation constraints: if the model somehow returns `rating=7`, Pydantic raises a validation error rather than silently letting bad data through. This is the same instinct as `TypedDict` for LangGraph state, which we'll meet shortly — make the shape of your data explicit so failures happen loudly and early.

### `with_structured_output` and LCEL composition

`llm.with_structured_output(ProductReview)` wraps the model so that, instead of returning an `AIMessage`, invoking it returns a *populated instance of `ProductReview`* directly. Under the hood, for most providers, this actually reuses the same tool-calling machinery from Section 2 — the schema is turned into a "tool" the model is forced to call, and the arguments it produces become your object. That's a nice symmetry: tools and structured output aren't two unrelated features, they're two applications of the same underlying capability.

`ChatPromptTemplate.from_messages([...])` builds a reusable prompt with named message roles (`system`, `user`) and `{placeholders}` you fill in later — this keeps the instruction ("you are a product review analyser…") separate from the changing input (the review text itself).

The `|` operator chains a prompt and a model into a single runnable — this is **LCEL** (LangChain Expression Language). `prompt | structure_llm` means "format the prompt, then pipe the result into the structured-output model." It's the same piping idea as Unix shell pipes.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import json

structure_llm = llm.with_structured_output(ProductReview)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a product review analyser. Extract structured information from reviews"),
    ("user", "{review_text}"),
])

chain = prompt | structure_llm

review_text = """
I bought this wireless mouse last month and it's been mostly great.
The battery life is incredible - I've only charged it once in 4 weeks.
The ergonomic design fits my hand perfectly and the buttons are responsive.
However, the scroll wheel is a bit stiff and makes clicking sounds.
Also, it's quite expensive compared to similar models.
Overall, I'd give it 4 out of 5 stars.
"""

response = chain.invoke({"review_text": review_text})

print(json.dumps(response.model_dump(), indent=2))

`response` here is a real `ProductReview` object — you can do `response.rating` or `response.pros[0]` with autocomplete and type-checking, not string-slicing. `.model_dump()` converts it to a plain dict (Pydantic v2's replacement for the old `.dict()`), which is what makes it JSON-serializable via `json.dumps`.

## 4. From one call to many — why you need an orchestrator

Sections 2 and 3 each augmented a *single* LLM call. But the content-generation task in your third script needs **four** LLM calls in a fixed sequence, where each one depends on the previous one's output:

1. Write a draft
2. Fact-check the draft
3. Revise the draft using the fact-check feedback
4. Format the final version for publication

You *could* write this as four sequential Python function calls with variables passed by hand. That works for four steps. It stops working cleanly once you need branching ("if the fact-check fails, loop back to drafting"), parallel steps, or you want to inspect/resume the process mid-way. **LangGraph** exists to make that structure explicit and manageable — this is the "prompt chaining" workflow pattern: decompose a task into fixed steps, where each LLM call processes the output of the previous one.

### State: a `TypedDict` as the shared contract

Every LangGraph graph is built around a **state** object — a single data structure that flows through every node, with each node reading what it needs and returning updates to merge in. Declaring it as a `TypedDict` is the same idea as the Pydantic model earlier: name every field up front so the "shape" of your workflow's data is explicit and typo-resistant (an IDE/type-checker will flag `state['topci']`; a plain dict would not).

In [ ]:
from typing import TypedDict

class ContentState(TypedDict):
    topic: str
    requirement: str
    draft: str
    fact_check_content: str
    improved_content: str
    final_draft: str

Each field corresponds to something produced by one of the four steps. By default, LangGraph's state update behavior is simple overwrite: when a node returns `{"draft": draft}`, it just sets `state["draft"]` to that value (fields the node doesn't mention are left untouched). This default is what lets each node below return only the one key it's responsible for, instead of the entire state.

### Nodes: plain functions, `state in → partial state out`

Every node in a LangGraph graph is just a Python function with the signature `(state) -> partial_state_update`. Here are your four, lightly cleaned up (the `print` progress messages are kept — handy for watching a multi-step chain execute).

In [ ]:
content_llm = ChatGroq(model="openai/gpt-oss-120b")

def generate_draft(state: ContentState) -> ContentState:
    """Generate initial blog post draft"""
    prompt = f"""Write a 200-word blog post about that: {state['topic']}

Requirements: {state['requirement']}
Focus on creating engaging, informative content."""

    draft = content_llm.invoke(prompt).content
    print("==== STEP 1: Draft Generated ====")
    print(draft[:150] + "...\n")
    return {"draft": draft}

**⚠️ Bug watch — a stray, oddly-indented line in the original prompt string:** in your source, the `f"""..."""` block for this prompt had `Focus on creating engaging, informative content` sitting flush against the left margin while every other line was indented — a classic multi-line-f-string formatting slip. It doesn't break anything (the model still receives the text; it's just also received a slightly ragged block of whitespace), but it's worth training your eye to catch, because ragged indentation inside triple-quoted prompt strings is one of the most common silent "my prompt looks fine on screen but has extra blank lines/leading spaces baked into it" bugs in LLM code.

In [ ]:
def fact_check(state: ContentState) -> ContentState:
    """Check draft for factual accuracy and consistency"""
    prompt = f"""Review the following blog post draft for factual accuracy and consistency:

{state['draft']}

Identify:
1. Any factual claims that seem questionable.
2. Internal inconsistencies
3. Statements that need citations

Provide a brief report."""

    fact_check_results = content_llm.invoke(prompt).content
    print("===== STEP 2: Fact Check Complete =====")
    print(fact_check_results[:150] + "....\n")
    return {"fact_check_content": fact_check_results}


def improve_content(state: ContentState) -> ContentState:
    """Revise content based on fact check feedback"""
    prompt = f"""Here is a blog post draft:
{state['draft']}

Here is feedback from fact-checking:
{state['fact_check_content']}

Revise the blog post to address the feedback while maintaining the engaging writing. Keep it around 200 words."""

    improved = content_llm.invoke(prompt).content
    print("===== STEP 3: Content Improved =======")
    print(improved[:150] + "...\n")
    return {"improved_content": improved}


def format_output(state: ContentState) -> ContentState:
    """Format content with HTML tags and elements"""
    prompt = f"""Format the following blog post for web publication:

{state['improved_content']}

Add:
- An engaging title wrapped in <h1> tags.
- Subheadings where appropriate with <h2>.
- Paragraph tags <p>.
- A meta description (1-2 sentences).
Output the formatted HTML."""

    final = content_llm.invoke(prompt).content
    print("==== STEP 4: Formatted for publication ====")
    return {"final_draft": final}

**⚠️ Bug watch — inconsistent logging:** the first three nodes each print a preview snippet of what they generated (`draft[:150]`, etc.); `format_output` prints only the step header, not a snippet of `final`. Not a functional bug — nothing downstream depends on the print statement — but it's an inconsistency you'd likely want to clean up if this were shared/production code, since it makes step 4's console output look broken by comparison.

**A conceptual note — is this really "quality control"?** Your file's docstring calls this a "content generation pipeline with quality control," but look closely at what `fact_check` actually does: it produces a text report, and `improve_content` folds that report back into a *rewrite* — there's no branching. The pipeline always runs draft → fact-check → improve → format, regardless of what the fact-check found. That's genuinely useful (feedback-informed revision), but it's a different pattern from a true quality **gate**, where you'd inspect the fact-check output and *conditionally* route: "if issues found, loop back to `generate_draft`; if clean, go straight to `format_output`." A gate needs a **conditional edge** — `builder.add_conditional_edges(...)` — with a routing function that decides the next node based on state, rather than the graph's fixed edges here. This is close to the **evaluator-optimizer** pattern (a generator and an evaluator loop until the evaluator is satisfied) — worth connecting to if you've built loops like that before, since the difference between "always revise once" and "revise until it passes" is exactly the difference between this script's linear chain and a graph with a cycle.

### Wiring the graph: nodes, edges, `START`/`END`, and compiling

`StateGraph(ContentState)` creates a graph builder typed to your state schema. `add_node` registers a function as a node — note that you can just pass the function object (`builder.add_node(generate_draft)`), and LangGraph uses `generate_draft.__name__` as the node's string name automatically, which is why the `add_edge` calls below can refer to `'generate_draft'` as a string even though it was never explicitly named. `START` and `END` are LangGraph's sentinel nodes marking where execution begins and terminates. `add_edge(a, b)` is a fixed, unconditional transition — always go from `a` straight to `b` — which is exactly what a linear chain needs (as opposed to `add_conditional_edges`, discussed above, for branching).

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ContentState)

builder.add_node(generate_draft)
builder.add_node(fact_check)
builder.add_node(improve_content)
builder.add_node(format_output)

builder.add_edge(START, 'generate_draft')
builder.add_edge('generate_draft', 'fact_check')
builder.add_edge('fact_check', 'improve_content')
builder.add_edge('improve_content', 'format_output')
builder.add_edge('format_output', END)

graph = builder.compile()

`.compile()` turns the builder into a runnable graph (it also does validation — e.g. it will complain about unreachable nodes or missing edges). The compiled `graph` exposes the same `.invoke()` interface you've been using on plain LLMs and chains all along — this consistency (everything is a "runnable" with `.invoke()`) is one of LangChain/LangGraph's core design choices, and it's why augmented LLMs and entire multi-step graphs compose so naturally.

In [ ]:
result = graph.invoke({
    "topic": "Physical Calisthenics",
    "requirement": "Target audience: busy professionals. Include practical tips",
})

print("\n" + "=" * 60)
print("Final Result")
print("=" * 50)
print(result["final_draft"])

`result` is the *final accumulated state* — a dict with every key `ContentState` declared, each filled in by whichever node last wrote to it. That's the payoff of the shared-state design: you don't have to manually thread `draft` → `fact_check` → `improved_content` through function return values; the graph does it, and you can inspect *any* intermediate value from `result`, not just the last one (`result["fact_check_content"]` is sitting right there too, useful for debugging even though the final print only uses `final_draft`).

## 5. Synthesis — how the three pieces fit together

Zooming out, all three scripts are facets of one idea:

- **`augmented_llm.py`** augments a single call with the ability to *act* (tools).
- **`augmented_llm_structured_output.py`** augments a single call with the ability to *report reliably* (schema-constrained output) — and, under the hood, leans on the same tool-calling mechanism as the first script.
- **`augmented_llm_prompt_chaining.py`** stops augmenting any *one* call and instead **orchestrates several calls** into a fixed workflow using LangGraph's state/node/edge model.

Nothing stops you from combining them: a LangGraph node's function body can itself call `llm.bind_tools(...)` or `llm.with_structured_output(...)` — a node is "just a function that touches state," and what happens inside is entirely up to you. In fact, a very natural next step for your content pipeline would be to make `fact_check` return a `with_structured_output` object like `FactCheckResult(has_issues: bool, notes: str)` instead of free text, so that a conditional edge could branch on `has_issues` cleanly instead of trying to parse a prose report. That single change would turn this from a prompt-chaining pipeline into something with a real quality gate — bridging directly into the evaluator-optimizer style pattern mentioned above.

The broader lesson: "agentic" behavior isn't one special technique — it's this same augmented-LLM building block, reused and composed, with the orchestration logic (chain, gate, loop, parallel fan-out) chosen to match the task's actual control-flow needs.

## 6. Exercises

Try these using the code above as a starting point:

1. **Close the tool-use loop.** Extend Section 2 so that after `llm_with_tools.invoke(weather_prompt)` returns a tool call, you actually execute `get_weather(...)` with the arguments the model chose, wrap the result in a `ToolMessage`, and send the conversation back to the model for a final natural-language answer.
2. **Fix the two bugs.** Rename `tip_percentge` → `tip_percentage` in the docstring, and normalize the `weather_data` dict so every entry uses the same unit consistently (pick C or F).
3. **Turn `fact_check` into structured output.** Define a small Pydantic model (e.g. `has_issues: bool`, `notes: str`) and make `fact_check` return that instead of prose, using `with_structured_output` from Section 3.
4. **Add a real quality gate.** Using `builder.add_conditional_edges`, write a routing function that sends the graph back to `generate_draft` when `has_issues` is `True` (from exercise 3), and on to `format_output` when it's `False`. Add a loop counter to `ContentState` so it can't loop forever.
5. **Add a tool to a graph node.** Give `generate_draft` access to a (fake) `search_recent_stats` tool it can call for supporting numbers, and thread the tool's result into the prompt before drafting.

## 7. Summary of concepts covered

- `.env` files + `load_dotenv()` for keeping API keys out of source code
- `ChatGroq` as one interchangeable implementation of LangChain's chat-model interface
- `@tool` decorator: turning a Python function (docstring + type hints) into an LLM-callable tool
- `bind_tools(...)` and reading `response.tool_calls`
- The gap between "the model chose a tool call" and "your code executed it and returned the result" (a full ReAct-style loop)
- Pydantic `BaseModel` + `Field(description=..., ge=..., le=...)` as a validated output schema
- `with_structured_output(...)` — schema-constrained generation, built on the same tool-calling mechanism
- `ChatPromptTemplate.from_messages([...])` and LCEL's `|` composition operator
- `.model_dump()` for converting a Pydantic object to a plain dict
- Why multi-step, dependent LLM calls need an orchestrator, not just sequential function calls
- LangGraph: `TypedDict` state, nodes as `(state) -> partial_update` functions, `add_node`, `add_edge`, `START`/`END`, `.compile()`, `.invoke()`
- Default state-update behavior (overwrite on the keys a node returns)
- The difference between a fixed linear chain (`add_edge`) and a conditional/looping workflow (`add_conditional_edges`) — i.e., prompt chaining vs. an evaluator-optimizer-style gate
- How tools, structured output, and graph orchestration are meant to be composed together, not used in isolation